### **CC-0F4 - Tópicos de Ciencia de la Computación I**

#### **Semana 1 - Laboratorio 1: de la atención causal a una evidencia defendible**

**Modalidad:** laboratorio guiado con trabajo individual y por parejas/grupos pequeños  
**Material base:** `Cuaderno1-CC-0F4.ipynb`, [Attention AI Lab](https://github.com/kapumota/attentionlab-ai), `annotated_deep_learning_paper_implementations/labml_nn/transformers/mha.py`, `annotated_deep_learning_paper_implementations/labml_nn/transformers/models.py`, *Attention Is All You Need*.


### **1. Propósito**

En la clase del lunes se estudió el paso desde un sistema de IA compuesto hacia uno de sus componentes: el Transformer. Se revisaron tokens, embeddings, estrategias posicionales, Q/K/V, scaled dot-product attention, máscara causal, Multi-Head Attention, residual, LayerNorm, FFN y las familias encoder-only, encoder-decoder y decoder-only.

Este laboratorio no repite esa teoría. El objetivo es producir evidencia de que el estudiante puede:

1. seguir una secuencia de tensores a través de un mecanismo de atención,
2. predecir y comprobar dimensiones,
3. identificar dónde aparece la causalidad,
4. verificar propiedades mediante métricas e invariantes,
5. introducir un fallo controlado y observar qué métrica lo detecta,
6. localizar la matemática dentro de una implementación real,
7. distinguir matemática, algoritmo, implementación, API y sistema,
8. formular una conclusión que no exceda la evidencia disponible,
9. defender oralmente una decisión técnica.


### **2. Ciclo metodológico del laboratorio**

Se mantienen las etiquetas en inglés porque funcionan como un **ciclo metodológico estable y reutilizable** durante CC-0F4:

```text
BUILD
  ->
EVALUATE
  ->
READ
  ->
CRITIQUE
  ->
DEFEND
```

Se interpretan así:

- **BUILD (construir):** recorre o implementa el componente y explica sus tensores.
- **EVALUATE (evaluar):** verifica propiedades con métricas e invariantes.
- **READ (leer):** localiza la teoría dentro de software y fuentes reales.
- **CRITIQUE (criticar técnicamente):** diseña una comparación controlada y limita la conclusión a la evidencia.
- **DEFEND (defender):** sustenta oralmente el razonamiento experimental.

La decisión de mantener las etiquetas en inglés es disciplinar: son nombres compactos de etapas que pueden reutilizarse cuando el curso pase de attention a retrieval, RAG, tools, agentes o multimodalidad, y conectan con workflows habituales de software y research engineering. Las instrucciones, explicaciones y evaluaciones permanecen en español.

> **Regla central:** que el código ejecute no demuestra que el componente sea correcto. Una afirmación técnica debe estar respaldada por una propiedad, una métrica o una comparación controlada.

### **3. Estándar experimental**

#### **Actividad 0.1 - Identificar la pregunta antes de ejecutar**

Considera las siguientes configuraciones:

- Configuración A: full self-attention.
- Configuración B: causal self-attention.

Completa antes de ejecutar código:

| Elemento | Respuesta del estudiante |
|---|---|
| Pregunta experimental | |
| Baseline | |
| Modificación | |
| Variable independiente | |
| Dos variables que deben mantenerse constantes | |
| Propiedad que espera que cambie | |
| Propiedad que podría permanecer correcta en ambas configuraciones | |
| Métrica principal | |

#### **Actividad 0.2 - Hipótesis falsable**

Escribe una hipótesis que pueda ser contradicha por los datos. Debe contener una comparación y una variable observable.

```text
Hipótesis:
```

No se acepta como hipótesis:

```text
"causal attention funciona"
```

La formulación debe indicar **qué se espera observar y bajo qué comparación**.


### **4. BUILD**

#### **Objetivo del bloque**

Construir conceptualmente y recorrer en el notebook la cadena:

```text
tokens
  ->
embeddings
  ->
Q, K, V
  ->
scores
  ->
scaling
  ->
causal mask
  ->
softmax
  ->
weighted values
  ->
heads
  ->
concatenación
  ->
proyección de salida
```

No se pide entrenar un Transformer.

La meta es demostrar que se comprende cómo una secuencia cambia de representación y de forma a través del mecanismo de atención.

#### **Ejercicio 1 - Pasaporte de tensores**

Trabaja inicialmente con:

$$
B=1,\qquad T=4,\qquad d_{\text{model}}=8,\qquad H=2.
$$

Calcula:

$$
d_{\text{head}}=\frac{d_{\text{model}}}{H}.
$$

Antes de ejecutar el notebook, completa las formas esperadas.

| Objeto | Forma predicha | Forma observada | Función del objeto |
|---|---|---|---|
| $X$ | | | |
| $Q$ | | | |
| $K$ | | | |
| $V$ | | | |
| $QK^\top$ | | | |
| máscara causal | | | |
| pesos de atención $A$ | | | |
| $AV$ por head | | | |
| concatenación de heads | | | |
| salida de MHA | | | |

In [ ]:
# Ejecuta esta celda solo después de completar la predicción de dimensiones.

import math
import torch
from torch import nn

torch.manual_seed(7)

B = 1
T = 4
d_model = 8
H = 2

assert d_model % H == 0
d_head = d_model // H

print({"B": B, "T": T, "d_model": d_model, "H": H, "d_head": d_head})

#### **Preguntas**

1. ¿Por qué $QK^\top$ contiene dos dimensiones asociadas a la secuencia?
2. ¿Qué dimensiones desaparecen o se reorganizan al concatenar los heads?
3. ¿Por qué la salida final de MHA debe volver a $d_{\text{model}}$?
4. Si $d_{\text{model}}=8$ y $H=4$, ¿cuánto vale $d_{\text{head}}$?
5. ¿Qué restricción debe satisfacer $d_{\text{model}}$ respecto de $H$ en esta implementación?.

#### **Ejercicio 2 - Reconstruir la atención producto punto escalado**

Localiza en el notebook las operaciones correspondientes a:

$$
Q=XW_Q,
$$

$$
K=XW_K,
$$

$$
V=XW_V.
$$

Después identifica la construcción de:

$$
S=QK^\top,
$$

$$
\widetilde{S}=\frac{QK^\top}{\sqrt{d_k}}.
$$

La atención causal se puede expresar como:

$$
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V,
$$

con:

$$
M_{ij}=
\begin{cases}
0, & j\le i,\\
-\infty, & j>i.
\end{cases}
$$

Completa:

| Expresión matemática | Línea/operación del notebook | Forma del tensor |
|---|---|---|
| $Q=XW_Q$ | | |
| $K=XW_K$ | | |
| $V=XW_V$ | | |
| $QK^\top$ | | |
| $1/\sqrt{d_k}$ | | |
| $+M$ | | |
| softmax | | |
| $AV$ | | |

In [ ]:
# BUILD - Completa/adapta a partir del Cuaderno3-MCC-corregido.ipynb.
# Mantén constantes las decisiones que no forman parte del ejercicio.

X = torch.randn(B, T, d_model)

W_Q = nn.Linear(d_model, d_model, bias=False)
W_K = nn.Linear(d_model, d_model, bias=False)
W_V = nn.Linear(d_model, d_model, bias=False)

# TODO: calcula Q, K y V.
Q = None
K = None
V = None

# TODO: reorganiza Q, K y V como [B, H, T, d_head].
# TODO: calcula QK^T.
# TODO: aplica 1/sqrt(d_head).
# TODO: construye y aplica la máscara causal.
# TODO: aplica softmax.
# TODO: multiplica los pesos por V.

#### **Ejercicio 3 - Rastrear una sola query**

Selecciona una query $i$ que no corresponda al primer ni al último token.

Registra $S_{i,:}$ antes del scaling, después $S_{i,:}/\sqrt{d_k}$, luego los scores con máscara y finalmente $A_{i,:}$.

| Etapa | Valores observados |
|---|---|
| Scores sin escalar | |
| Scores escalados | |
| Scores después de máscara | |
| Pesos después de softmax | |

Comprueba:

$$
\sum_j A_{ij}\approx 1.
$$

Y para toda posición futura:

$$
j>i \Longrightarrow A_{ij}\approx 0.
$$

In [ ]:
# Selecciona una query intermedia y registra cada etapa.
query_index = 2

# TODO:
# row_scores = ...
# row_scaled = ...
# row_masked = ...
# row_attention = ...

# print("puntuaciones:", row_scores)
# print("escalado:", row_scaled)
# print("enmascarado:", row_masked)
# print("atención:", row_attention)
# print("suma de filas:", row_attention.sum())

#### **Preguntas**

1. ¿La máscara se aplica antes o después del softmax? ¿Por qué importa?
2. ¿Qué ocurriría si las posiciones futuras se reemplazaran por $0$ en vez de $-\infty$ antes del softmax?
3. ¿Softmax, por sí solo, introduce causalidad?
4. ¿Qué información aporta $V$ que no aporta la matriz de compatibilidad $QK^\top$?.

#### **Ejercicio 4 - Atención multicabecera**

Con $H=2$, selecciona una misma query $i$ y compara:

$$
A^{(1)}_{i,:}
\qquad\text{y}\qquad
A^{(2)}_{i,:}.
$$

| Head | Distribución de atención para la query seleccionada | Índice de mayor peso |
|---|---|---|
| 1 | | |
| 2 | | |

Responde:

1. ¿Deben producir ambos heads exactamente la misma distribución? Justifica.
2. ¿Qué cambia entre heads si todos reciben la misma secuencia de entrada?
3. ¿Qué significa concatenar $\operatorname{head}_1,\ldots,\operatorname{head}_H$ antes de aplicar $W_O$?
4. Explica por qué observar patrones diferentes entre heads **no permite concluir** que un head "entiende sintaxis" y otro "entiende semántica".

La salida de MHA se expresa como:

$$
\operatorname{MHA}(X)=
\operatorname{Concat}(\operatorname{head}_1,\ldots,\operatorname{head}_H)W_O.
$$

Identifica en el notebook dónde ocurre la concatenación y dónde ocurre la proyección de salida.


In [ ]:
# TODO: compara una misma query entre al menos dos heads.
# Usa la atención obtenida con tu implementación o con el cuaderno corregido.

# attn_head_1 = ...
# attn_head_2 = ...

# print(attn_head_1)
# print(attn_head_2)

### **5. Attention AI Lab**

#### **Objetivo del bloque**

Conectar:

```text
matemática
  ->
implementación
  ->
API
  ->
frontend
  ->
sistema
```

No se evalúa conocimiento de FastAPI.

#### **Ejercicio 5 - Modo Basic: predicción antes de observar**

Abre el módulo de atención.

#### **Parte A - Full self-attention**

Antes de ejecutar, responde:

1. Para una query $i$, ¿está permitido que exista $A_{ij}>0$ cuando $j>i$?
2. Dibuja el patrón cualitativo que espera observar en una matriz $T\times T$.

#### **Parte B - Causal self-attention**

Antes de cambiar de modo, dibuja el patrón esperado e indica explícitamente qué región debe satisfacer $A_{ij}=0$.

Después observa el resultado.

| Modo | Predicción | Observación | ¿Coinciden? |
|---|---|---|---|
| Full attention | | | |
| Causal attention | | | |

#### **Ejercicio 6 - Modo Technical: conectar fórmula y control**

Relaciona cada concepto visible en la interfaz con:

$$
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V.
$$

| Elemento | Papel matemático | Efecto esperado |
|---|---|---|
| scores | | |
| scale | | |
| causal mask | | |
| softmax | | |
| attention matrix | | |

Responde:

1. Full attention y causal attention pueden utilizar softmax. ¿Qué explica entonces la diferencia estructural entre ambas matrices?
2. ¿Una matriz visualmente triangular basta para demostrar corrección? Justifica.

#### **Ejercicio 7 - Modo Expert: del algoritmo a la API**

Abre la vista experta correspondiente a:

```text
POST /api/attention/compute
```

Sin modificar el backend, identifica:

| Evidencia | Valor/observación |
|---|---|
| Endpoint | |
| Método HTTP | |
| Campos principales del request | |
| Campos principales del response | |
| Información relacionada con atención | |
| Dato que utiliza el frontend para visualizar el resultado | |

Clasifica:

| Elemento | Nivel |
|---|---|
| $Q,K,V$ | |
| función que calcula atención | |
| `/api/attention/compute` | |
| request JSON | |
| response JSON | |
| matriz mostrada por React | |
| Attention AI Lab completo | |

Usa únicamente estas categorías:

```text
matemática
implementación
interfaz
contrato
dato serializado
presentación
sistema
```

#### **Pregunta de ingeniería**

Explica en un párrafo:

> ¿Qué información debe atravesar la frontera entre backend y frontend para que una propiedad matemática como la causalidad pueda ser observada por el usuario?.


### **6. EVALUATE**

#### **Ejercicio 8 - Invariante de normalización**

Calcula:

$$
E_{\text{norm}}
=
\max_i\left|\sum_j A_{ij}-1\right|.
$$

Registra:

```text
E_norm =
```

Define un umbral numérico que consideres aceptable y justifica por qué se necesita una tolerancia en punto flotante.

In [ ]:
def normalization_error(attn: torch.Tensor):
    # TODO: calcula el máximo error absoluto de normalización por fila.
    # Decide explícitamente qué dimensión corresponde a las keys.
    pass

#### **Ejercicio 9 - Invariante de causalidad**

Calcula:

$$
L_{\text{future}}
=
\max_{j>i} A_{ij}.
$$

Registra:

```text
L_future =
```

Explica qué propiedad del componente intenta medir.

In [ ]:
def future_attention_leakage(attn: torch.Tensor):
    # TODO: calcula el máximo peso asignado a posiciones futuras j > i.
    # Considera batch y heads si están presentes.
    pass

#### **Ejercicio 10 - Fallo controlado**

Ejecuta dos configuraciones manteniendo constantes input, pesos, número de heads, $d_{\text{model}}$, seed y cualquier otra variable no relacionada con la modificación.

In [ ]:
# EVALUATE - Mantén todas las variables constantes excepto causal.

# TODO:
# attn_causal = ...
# attn_full = ...

# TODO: calcula E_norm y L_future para ambas configuraciones.

results = {
    "causal": {"E_norm": None, "L_future": None},
    "full": {"E_norm": None, "L_future": None},
}
results

#### Baseline

```text
causal=True
```

#### Intervención

```text
causal=False
```

Antes de ejecutar escribe:

```text
Predicción sobre E_norm:
Predicción sobre L_future:
```

Después completa:

| Configuración | $E_{\text{norm}}$ | $L_{\text{future}}$ | ¿Normaliza? | ¿Es causal? |
|---|---:|---:|---|---|
| causal | | | | |
| full | | | | |

#### **Preguntas**

1. ¿Qué métrica detectó el fallo?
2. ¿Qué métrica pudo permanecer correcta?
3. ¿Por qué una única métrica no basta para afirmar que el componente es correcto?
4. ¿Qué propiedad fue alterada realmente al cambiar `causal=True` por `causal=False`?
5. Formula una conclusión que no exceda la evidencia disponible.



### **7. READ**

#### **Objetivo del bloque**

Localizar dentro de software real los elementos matemáticos estudiados.

Material:

```text
annotated_deep_learning_paper_implementations/
  labml_nn/transformers/mha.py

annotated_deep_learning_paper_implementations/
  labml_nn/transformers/models.py
```

No se pide comprender todo el repositorio.

#### **Ejercicio 11 - `mha.py`: paper -> ecuación -> código**

Localiza sin usar búsqueda automática las partes correspondientes a:

$$
Q=XW_Q,\quad K=XW_K,\quad V=XW_V,
$$

$$
S=QK^\top,
$$

$$
\widetilde{S}=\frac{S}{\sqrt{d_k}},
$$

$$
A=\operatorname{softmax}(\widetilde{S}+M),
$$

$$
O=AV,
$$

$$
\operatorname{MHA}
=
\operatorname{Concat}(\operatorname{head}_1,\ldots,\operatorname{head}_H)W_O.
$$

Completa:

| Matemática | Clase/función/línea conceptual en `mha.py` | Forma o comentario relevante |
|---|---|---|
| $Q$ | | |
| $K$ | | |
| $V$ | | |
| $QK^\top$ | | |
| $1/\sqrt{d_k}$ | | |
| máscara | | |
| softmax | | |
| $AV$ | | |
| concatenación de heads | | |
| $W_O$ | | |

#### **Preguntas**

1. ¿Por qué `einsum` necesita hacer explícitos índices que en el paper aparecen ocultos en $QK^\top$?
2. ¿La clase `MultiHeadAttention` es causal por sí misma? Justifica a partir de su interfaz.
3. ¿Qué información conserva `self.attn` y por qué podría ser útil para evaluación o depuración?
4. Diferencia una decisión requerida por la matemática de una decisión de ingeniería de software presente en el archivo.

#### **Ejercicio 12 - `models.py`: construir un `TransformerLayer`**

Localiza:

1. LayerNorm antes de self-attention.
2. self-attention.
3. primera conexión residual.
4. cross-attention opcional.
5. LayerNorm antes del FFN.
6. FFN.
7. segunda conexión residual.

Relaciona el código con:

$$
Y=X+\operatorname{MHA}(\operatorname{LN}(X)),
$$

$$
Z=Y+\operatorname{FFN}(\operatorname{LN}(Y)).
$$

Completa:

| Etapa | Fragmento conceptual de `models.py` | ¿Mezcla tokens? | ¿Transforma características? |
|---|---|---|---|
| LayerNorm | | | |
| self-attention | | | |
| residual | | | |
| FFN | | | |

#### **Pregunta avanzada**

El mismo `TransformerLayer` admite `src_attn` opcional. Explica cómo esta decisión de diseño permite reutilizar la clase tanto en capas encoder como decoder sin duplicar toda la implementación.


### **8. Exposición modelo**

#### **Actividad 13 - Observar cómo se presenta un paper**

Durante la exposición del docente sobre *Attention Is All You Need*, no tome notas cronológicas. Completa únicamente esta ficha:

| Elemento | Nota del estudiante |
|---|---|
| Problema que intenta resolver el paper | |
| Idea central | |
| Mecanismo técnico principal | |
| Una ecuación o construcción indispensable | |
| Evidencia utilizada por los autores | |
| Una limitación | |
| Una decisión que hoy podría cambiarse | |
| Conexión con el código estudiado | |

#### **Pregunta**

Escribe una diferencia concreta entre:

```text
resumir un paper
```

y:

```text
analizar técnicamente un paper
```

### **9. CRITIQUE**

#### **Objetivo del bloque**

Diseñar y ejecutar un experimento pequeño cuya conclusión sea proporcional a la evidencia.

Trabaja en grupos pequeños y escoja **una sola** línea experimental.

#### **Opción A - Full attention vs causal attention**

Pregunta: ¿Qué propiedad cambia al introducir una máscara causal?

Define:

```text
Baseline:
Intervención:
Variables constantes:
Métrica principal:
Métrica secundaria:
Hipótesis:
```

Incluye al menos:

$$
L_{\text{future}}=\max_{j>i}A_{ij}.
$$

No concluyas que una variante es "mejor" sin definir previamente una tarea y una métrica de calidad.

#### **Opción B - Efecto del scaling**

Compara:

$$
S=QK^\top
$$

contra:

$$
\widetilde{S}=\frac{QK^\top}{\sqrt{d_k}}.
$$

Mantén constantes $Q$ y $K$.

Puede usar:

$$
p_{\max}=\max_j A_{ij},
$$

 y/o:

$$
H(A_i)=-\sum_j A_{ij}\log A_{ij}.
$$

Completa antes de ejecutar:

```text
Hipótesis:
Baseline:
Intervención:
Variable independiente:
Variables constantes:
Métrica:
Resultado esperado:
```

Después registra al menos dos queries.

| Query | Sin scaling: $p_{\max}$ | Con scaling: $p_{\max}$ | Sin scaling: entropía | Con scaling: entropía |
|---|---:|---:|---:|---:|
| 1 | | | | |
| 2 | | | | |

No generalices el resultado del ejemplo pequeño a todos los Transformers.

In [ ]:
def row_entropy(probabilities: torch.Tensor, eps: float = 1e-12):
    # TODO: implementa la entropía de una distribución de atención.
    pass

# TODO:
# selecciona al menos dos queries;
# compara p_max y entropía con y sin scaling;
# conserva Q y K idénticos.

#### **Opción C - Número de heads**

Mantén $d_{\text{model}}$ constante y compara al menos $H=1$ y $H=2$. Si la implementación lo permite, añade $H=4$.

| $H$ | $d_{\text{head}}$ | Forma de $Q$ | Forma de $A$ | Forma tras concatenación |
|---:|---:|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 4 | | | | |

#### **Preguntas**

1. ¿Qué cambia cuando aumenta $H$ manteniendo $d_{\text{model}}$ constante?
2. ¿Qué permanece constante?
3. ¿El experimento demuestra que un número mayor de heads produce mejor calidad? Justifica.
4. ¿Qué experimento adicional sería necesario para estudiar calidad y no solo estructura?.

#### **Hoja experimental obligatoria**

Todos los estudiantes o grupos deben completar:

```text
Pregunta:

Hipótesis:

Baseline:

Intervención:

Variable independiente:

Variables constantes:

Métrica(s):

Resultado(s):

¿La hipótesis fue apoyada o contradicha por el resultado?:

Conclusión:

Una limitación del experimento:

Una afirmación que NO puede defenderse con esta evidencia:
```

### **10. DEFEND**

#### **Actividad 14 - Defensa oral breve**

El docente seleccionará grupos e integrantes de manera aleatoria.

Responde, sin leer el notebook:

1. ¿Qué pregunta intentaron responder?
2. ¿Cuál fue el baseline?
3. ¿Qué modificaron?
4. ¿Qué mantuvieron constante?
5. ¿Qué métrica utilizaron?
6. ¿Qué resultado obtuvieron?
7. ¿Qué conclusión pueden defender?
8. ¿Qué conclusión no pueden defender?
9. ¿Qué limitación tiene el experimento?

No se evalúa la memoria literal del código. Se evalúa la relación:

```text
pregunta
  ->
diseño experimental
  ->
evidencia
  ->
conclusión
```


### **11. Evidencia mínima de entrega**

La entrega debe contener evidencia de BUILD, Attention AI Lab, EVALUATE, READ, CRITIQUE y, cuando sea solicitado, DEFEND.

#### **BUILD**

- tabla de formas predichas y observadas,
- rastreo de una query,
- comparación de al menos dos heads,
- explicación de dónde entra la máscara causal.

#### **Attention AI Lab**

- predicción full vs causal,
- observación,
- tabla matemática -> implementación -> API -> frontend -> sistema.

#### **EVALUATE**

- $E_{\text{norm}}$,
- $L_{\text{future}}$,
- tabla causal vs full,
- fallo controlado;
- conclusión.

#### **READ**

- tabla ecuación -> código de `mha.py`;
- identificación del bloque pre-norm en `models.py`.

#### **CRITIQUE**

- hoja experimental completa,
- valores obtenidos,
- una limitación,
- una afirmación que la evidencia no permite sostener.

#### **DEFEND**

- defensa oral individual si el docente la solicita.


### **12. Reglas de trabajo**

1. Puedes utilizar asistentes generativos para estudiar, programar, depurar o explorar alternativas.
2. Debes poder defender todo código, resultado o explicación que presentes.
3. No modifiques más de una variable cuando el objetivo sea atribuir un efecto a una intervención concreta.
4. No concluyas "mejor" o "peor" sin una métrica que represente la propiedad de interés.
5. Diferencia siempre resultado observado de interpretación.
6. Una visualización sirve como evidencia complementaria, no sustituye una comprobación numérica cuando existe una invariante verificable.
7. Mantén seeds y configuraciones relevantes registradas.
8. Attention AI Lab se utiliza como herramienta didáctica, no como benchmark de rendimiento de un LLM real.
9. No necesitas RAG, agentes, KV cache ni multimodalidad en este laboratorio.
10. En la defensa oral, la incapacidad de explicar una parte esencial de la entrega invalida esa parte como evidencia de comprensión.


### **13. Criterio de cierre**

Al terminar el laboratorio, el estudiante debe poder sostener técnicamente:

$$
X
\longrightarrow
Q,K,V
\longrightarrow
\frac{QK^\top}{\sqrt{d_k}}
\longrightarrow
+M
\longrightarrow
\operatorname{softmax}
\longrightarrow
AV
\longrightarrow
\operatorname{MHA}.
$$

Además debe poder explicar por qué:

$$
M_{ij}=-\infty\qquad \text{para }j>i
$$

produce:

$$
A_{ij}\approx 0\qquad \text{para }j>i,
$$

y cómo verificarlo mediante:

$$
L_{\text{future}}=\max_{j>i}A_{ij}.
$$

El resultado final del laboratorio no es "el notebook ejecutó".

El resultado esperado es una afirmación del tipo:

> **Bajo una configuración registrada, modificamos una propiedad concreta, la medimos con una métrica apropiada, observamos un resultado reproducible, identificamos una limitación y podemos defender qué conclusión está respaldada por esa evidencia.**

Ese es el estándar experimental de CC-0F4.